# ResQue: Warm-Start QRC for Multi-Output Weather Forecasting over East Africa
### ResQue | GIC 2026 | Track B: Weather Time-Series Forecasting

[![Launch on qBraid](https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png)](https://account.qbraid.com?gitHubUrl=https://github.com/Armstrong66/resque-qrc)

**This notebook is the Phase 3 entry point. Run cells sequentially.**  
All results written to `outputs/results/`. Full benchmark tables match the write-up.

**Cells 1–12 are 100% simulator** (free, no QPU credits used) — that's where
the Hamiltonian/noise/qubit/shot sweeps and baseline training happen. Real
QPU credits are spent ONLY in **Cell 13**, which validates the final
selected config over a small subsampled window — see that cell before
running it.

| Setting | Value |
|---|---|
| Station | NOAA ISD 63450099999 (Addis Ababa Bole, Ethiopia) |
| Horizons | 6h and 24h |
| Primary reservoir | 9-qubit transverse-field Ising chain |
| Encoding ablation | Standard vs. data reuploading (Pérez-Salinas et al. 2020) |
| Warm-start | ESN / LSTM / GRU → QRC via truncated SVD (config.WARM_START_SOURCE) |
| Hardware (Cell 13 only) | QuEra Aquila (PRIMARY — real analog Rydberg mapping, validated against free local emulators, not yet run on live hardware) / IBM Eagle-Heron (FALLBACK — real gate-based device swap, not yet hardware-verified) — see `reservoir/aquila_backend.py` and `docs/PROJECT_CRITIQUE.md` for exactly what "validated" means here |

**AI disclosure**: Claude (Anthropic) used for code scaffolding — disclosed per GIC rules.

---
## Cell 1 — Environment setup

In [ ]:
# ── Install any missing dependencies ──────────────────────────────────────────
# qBraid Lab has PennyLane, torch, numpy, pandas pre-installed.
# statsmodels/pmdarima, pyarrow, and bloqade-analog may need installing.
# NOTE: statsmodels is installed FIRST and separately — it is the ARIMA
# fallback backend (see baselines/classical.py) and must not silently be
# skipped if the pmdarima install fails on a given environment.
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['statsmodels', 'pmdarima', 'pyarrow', 'pennylane-lightning']:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        try:
            _install(pkg)
        except Exception as e:
            print(f'  {pkg} install failed ({e}) — continuing; '
                  f'see baselines/classical.py for fallback behaviour')

# bloqade-analog (PyPI name) imports as `bloqade.analog`, not `bloqade_analog`
# — handled separately since the generic name-mangling check above doesn't
# apply. This is the PRIMARY hardware path (QuEra Aquila, see Cell 13);
# installing it here means Cell 13 doesn't need its own install step.
try:
    import bloqade.analog  # noqa: F401
except ImportError:
    print('Installing bloqade-analog...')
    try:
        _install('bloqade-analog')
    except Exception as e:
        print(f'  bloqade-analog install failed ({e}) — Cell 13\'s Aquila '
              f'backend will be unavailable until this is resolved.')

# ── Confirm versions ──────────────────────────────────────────────────────────
import pennylane as qml
import torch
import numpy as np
import pandas as pd

print(f'PennyLane : {qml.__version__}')
print(f'PyTorch   : {torch.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
print(f'NumPy     : {np.__version__}')

try:
    import bloqade.analog as _bloqade
    print(f'Bloqade   : {_bloqade.__version__} (Aquila path available)')
except ImportError:
    print('Bloqade   : NOT installed — Cell 13 Aquila backend will be unavailable')

# ── Add project root to path ──────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'\nProject root: {PROJECT_ROOT}')
print('Environment ready.')

---
## Cell 2 — Configuration
All hyperparameters live in `config.py`. Override here for quick experiments.

In [ ]:
from config import *

# ── Override defaults for Phase 3 full run ────────────────────────────────────
# Change these to reproduce specific results from the write-up.

RUN_MODE = 'full'          # 'smoke' (fast, 300 steps) | 'full' (complete dataset)
N_QUBITS_PRIMARY = 9       # Primary result qubit count
J_STAR  = 0.3              # From Phase 2 Hamiltonian sweep — RE-SWEEP after changing
H_STAR  = 0.8              # USE_REUPLOADING; J*/h* are encoding-dependent (see
                            # best_hamiltonian.json's "use_data_reuploading" field).
P_STAR  = 0.0              # From Phase 2 noise sweep (noiseless optimal in sim)
TOPOLOGY = 'chain'         # chain outperformed all_to_all in Phase 2
USE_REUPLOADING = True     # Phase 3 primary encoding — set False for standard ablation
WARM_START_SOURCE = 'esn'  # 'esn' | 'lstm' | 'gru' — one-line switch, see config.py

SWEEP_STEPS = 300 if RUN_MODE == 'smoke' else None  # None = full dataset

print(f'Run mode         : {RUN_MODE}')
print(f'Qubits (primary) : {N_QUBITS_PRIMARY}')
print(f'J*, h*           : {J_STAR}, {H_STAR}')
print(f'Data reuploading : {USE_REUPLOADING}')
print(f'Warm-start source: {WARM_START_SOURCE}')
print(f'Sweep steps      : {"full dataset" if SWEEP_STEPS is None else SWEEP_STEPS}')

---
## Cell 3 — Data download
NOAA ISD global-hourly archive. No API key required.

In [ ]:
from data.downloader import download_all

raw_paths = download_all()
print(f'\n{len(raw_paths)} year files ready.')

---
## Cell 4 — Parse, clean, and inspect
ISD sentinel replacement → Magnus RH → 6h resample → ffill/bfill → dropna

In [ ]:
from data.parser import load_and_merge

df = load_and_merge(raw_paths)

print(f'Timesteps  : {len(df)}')
print(f'Date range : {df.index.min()} → {df.index.max()}')
print(f'\nNaN check (must all be 0%):')
for col in TARGETS:
    pct = df[col].isna().mean() * 100
    flag = '✓' if pct == 0 else f'✗ {pct:.1f}% — DELETE PARQUET AND RERUN'
    print(f'  {col:<20} {flag}')

print(f'\nDescriptive stats (physical units):')
display(df[TARGETS].describe().round(2))

---
## Cell 5 — Preprocessing: shared PCA + windowing

In [ ]:
from preprocessing.pipeline import WeatherPreprocessor

if RUN_MODE == 'smoke':
    df_use = df.iloc[:500]
    print('Smoke mode: using first 500 timesteps')
else:
    df_use = df

prep = WeatherPreprocessor(df_use)
datasets = prep.build_all()
prep.save(datasets)

for h, ds in datasets.items():
    print(ds.summary())

ds6  = datasets[6]
ds24 = datasets[24]
print('\nDatasets ready.')

---
## Cell 6 — Classical baselines
Persistence → ARIMA → ESN → LSTM → GRU  
ESN also provides warm-start weights for QRC readout.

In [ ]:
from baselines.classical import (run_persistence, run_arima,
                                  run_esn, run_rnn, ARIMA_AVAILABLE, TORCH_AVAILABLE)
import time

baseline_results = {}
fitted_esn = None
fitted_rnn = {}
X_train_warm = None   # source picked below, matches config.WARM_START_SOURCE

if not ARIMA_AVAILABLE:
    print('WARNING: no ARIMA backend installed (statsmodels/pmdarima) — '
          'ARIMA will be ABSENT from results. pip install statsmodels pmdarima')
if not TORCH_AVAILABLE:
    print('WARNING: PyTorch not installed — LSTM/GRU will be ABSENT from results.')

# Persistence
r = run_persistence(ds6.y_val, ds6.y_test, ds6.X_val, ds6.X_test, window=WINDOW_SIZE)
baseline_results['persistence'] = r
print(f'Persistence  test RMSE (mean): {r.test_rmse.mean():.4f}')

# ARIMA (backend: pmdarima if installed, else statsmodels grid-search fallback —
# see baselines/classical.py ARIMA_AVAILABLE)
r = run_arima(ds6.y_train, ds6.y_val, ds6.y_test, TARGETS)
if r:
    baseline_results['arima'] = r
    print(f"ARIMA        test RMSE (mean): {r.test_rmse.mean():.4f}  [backend={r.meta.get('backend')}]")
else:
    print('ARIMA        SKIPPED — no backend available')

# ESN — CRITICAL baseline; also the default warm-start source
t0 = time.time()
r_esn, fitted_esn = run_esn(ds6.X_train, ds6.y_train,
                              ds6.X_val,   ds6.y_val,
                              ds6.X_test,  ds6.y_test)
baseline_results['esn'] = r_esn
print(f'ESN          test RMSE (mean): {r_esn.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

# LSTM — run_rnn returns (result, warm_start_extractor)
t0 = time.time()
r, wrapper = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val,
                      ds6.X_test,  ds6.y_test,  window=WINDOW_SIZE, model_type='lstm')
if r:
    baseline_results['lstm'] = r
    fitted_rnn['lstm'] = wrapper
    print(f'LSTM         test RMSE (mean): {r.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

# GRU
t0 = time.time()
r, wrapper = run_rnn(ds6.X_train, ds6.y_train, ds6.X_val, ds6.y_val,
                      ds6.X_test,  ds6.y_test,  window=WINDOW_SIZE, model_type='gru')
if r:
    baseline_results['gru'] = r
    fitted_rnn['gru'] = wrapper
    print(f'GRU          test RMSE (mean): {r.test_rmse.mean():.4f}  [{time.time()-t0:.1f}s]')

# Warm-start source — configurable via config.WARM_START_SOURCE ("esn"|"lstm"|"gru").
# ARIMA is not a valid source (no reservoir-like hidden state).
if WARM_START_SOURCE == 'esn' and fitted_esn is not None:
    X_train_warm = fitted_esn.get_reservoir_states(ds6.X_train)
elif WARM_START_SOURCE in ('lstm', 'gru') and fitted_rnn.get(WARM_START_SOURCE) is not None:
    X_train_warm = fitted_rnn[WARM_START_SOURCE].get_hidden_states(ds6.X_train)
print(f"\nWarm-start source: {WARM_START_SOURCE}  (states shape: "
      f"{None if X_train_warm is None else X_train_warm.shape})")


---
## Cell 7 — Encoding ablation: standard vs. data reuploading
**Phase 3 primary experiment.** Compares single-injection vs. re-encoding at every Trotter step.

In [ ]:
from reservoir.quantum_reservoir import encoding_ablation

encoding_results = encoding_ablation(
    X_train=ds6.X_train, y_train=ds6.y_train,
    X_val=ds6.X_val,     y_val=ds6.y_val,
    n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
    max_steps=SWEEP_STEPS,
    out_dir=RESULTS
)

print(f"\nEncoding ablation (n={N_QUBITS_PRIMARY} qubits, J={J_STAR}, h={H_STAR}):")
for enc, rmse in encoding_results.items():
    print(f"  {enc:<22} val_rmse = {rmse:.4f}")

improvement = encoding_results.get('standard', 0) - encoding_results.get('data_reuploading', 0)
print(f"\n  Reuploading improvement: {improvement:+.4f} (positive = reuploading wins)")

---
## Cell 8 — QRC training: cold-start and warm-start
Uses the encoding strategy selected above.

In [ ]:
from reservoir.quantum_reservoir import IsingQRC
from readout.ridge_readout import RidgeReadout
import json, pickle

qrc_results = {}
out_h6 = RESULTS / 'h6'
out_h6.mkdir(parents=True, exist_ok=True)

for label, use_warm in [('cold_start_qrc', False), ('warm_start_qrc', True)]:
    print(f'\n--- Training: {label} ---')
    t0 = time.time()
    try:
        qrc = IsingQRC(
            n_qubits=N_QUBITS_PRIMARY, J=J_STAR, h=H_STAR,
            topology=TOPOLOGY, noise_rate=P_STAR,
            use_data_reuploading=USE_REUPLOADING
            # hardware_backend defaults to "simulation" here — this cell (and
            # the sweeps in Cells 9/11/12) are ALWAYS simulator, by design.
            # Real hardware happens ONLY in Cell 13, on a small subsampled
            # window of the config selected here — see that cell for why.
        )

        H_train = qrc.run_sequence(ds6.X_train, verbose=True)
        H_val   = qrc.run_sequence(ds6.X_val)
        H_test  = qrc.run_sequence(ds6.X_test)

        n_tr = min(len(H_train), len(ds6.y_train))
        n_vl = min(len(H_val),   len(ds6.y_val))
        n_ts = min(len(H_test),  len(ds6.y_test))

        readout = RidgeReadout(
            target_names=TARGETS,
            warm_start=(use_warm and X_train_warm is not None)
        )
        best = readout.fit(
            H_train[:n_tr], ds6.y_train[:n_tr],
            H_val[:n_vl],   ds6.y_val[:n_vl],
            X_train_warm_start=X_train_warm[:n_tr] if X_train_warm is not None else None
        )
        readout.save_selection_log(out_h6)

        pred_test = best.predict(H_test[:n_ts])
        pred_val  = best.predict(H_val[:n_vl])

        # Store for metrics table
        class _R:
            def __init__(self, pv, pt):
                self.y_pred_val  = pv
                self.y_pred_test = pt
        qrc_results[label] = _R(pred_val, pred_test)

        # Persist BOTH config and the fitted readout weights, under the same
        # outputs/results/h{horizon}/ layout main.py and agent_runner.py use —
        # Cell 13's hardware validation (and scripts/hardware_validation.py /
        # agent_runner.py --task hardware_validation run standalone later)
        # reads exactly these two files. Previously this cell only saved the
        # config, and saved it at the wrong (top-level, not per-horizon) path,
        # so hardware validation could never find a fitted readout to reuse.
        best.save(out_h6 / f'{label}_readout.pkl')
        cfg = qrc.get_config()
        cfg.update({'horizon_hours': 6, 'readout_strategy': best.strategy,
                    'warm_start': use_warm, 'warm_start_source': WARM_START_SOURCE,
                    'shared_pca': USE_SHARED_PCA,
                    'val_rmse_mean': float(best.val_rmse_mean),
                    'wall_clock_s': round(time.time()-t0, 1)})
        with open(out_h6 / f'{label}_config.json', 'w') as f:
            json.dump(cfg, f, indent=2)

        print(f'  Done. Strategy={best.strategy} val_rmse={best.val_rmse_mean:.4f} '
              f'[{time.time()-t0:.0f}s]')

    except Exception as e:
        import traceback
        print(f'  FAILED: {e}')
        traceback.print_exc()

---
## Cell 9 — Qubit scaling study
Required by challenge: characterise performance across n = 5 → 20.

In [ ]:
from experiments.sweeps import qubit_scaling_study

df_scaling = qubit_scaling_study(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR,
    qubit_counts=QUBIT_COUNTS,   # [5, 7, 9, 12, 16, 20]
    use_data_reuploading=USE_REUPLOADING
)

print('\nQubit scaling results:')
display(df_scaling[['n_qubits', 'feature_dim', 'val_rmse']].to_string(index=False))

---
## Cell 10 — Full benchmark table
All models, both horizons, RMSE + MAE in physical units + VPT.

In [ ]:
from evaluation.metrics import build_results_table

all_results = {**baseline_results, **qrc_results}

print('=== 6-HOUR HORIZON ===')
df_6h = build_results_table(
    results=all_results,
    y_true_val=ds6.y_val,
    y_true_test=ds6.y_test,
    target_names=TARGETS,
    horizon_hours=6,
    out_dir=RESULTS
)
display(df_6h)

# ── Repeat for 24h horizon ────────────────────────────────────────────────────
print('\n=== 24-HOUR HORIZON ===')
# Re-run QRC on ds24 if full run mode
if RUN_MODE == 'full':
    print('(24h QRC run — this will take a while, same as Cell 8 but on ds24)')
    # Add 24h QRC here following same Cell 8 pattern with ds24

df_24h = build_results_table(
    results=baseline_results,  # Add 24h QRC results when available
    y_true_val=ds24.y_val,
    y_true_test=ds24.y_test,
    target_names=TARGETS,
    horizon_hours=24,
    out_dir=RESULTS
)
display(df_24h)

---
## Cell 11 — Noise sweep
Tests whether hardware-induced noise improves generalisation (Antoncich et al. 2026).

In [ ]:
from experiments.sweeps import noise_sweep

p_star, df_noise = noise_sweep(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, n_qubits=N_QUBITS_PRIMARY,
    use_data_reuploading=USE_REUPLOADING
)

print(f'\nOptimal noise rate p* = {p_star}')
display(df_noise)

noiseless = df_noise[df_noise.noise_rate == 0.0].val_rmse.values[0]
if p_star > 0:
    best_noisy = df_noise[df_noise.noise_rate == p_star].val_rmse.values[0]
    print(f'Noise-assisted: {noiseless:.4f} (p=0) → {best_noisy:.4f} (p={p_star})')
else:
    print(f'Noiseless optimal: {noiseless:.4f}. Hardware test may differ.')

---
## Cell 12 — Shot budget ablation
Confirms RMSE stability from exact simulation through 500→5000 shots.

In [ ]:
from experiments.sweeps import shot_ablation

df_shots = shot_ablation(
    ds6.X_train, ds6.y_train,
    ds6.X_val,   ds6.y_val,
    J=J_STAR, h=H_STAR, p=P_STAR, n_qubits=N_QUBITS_PRIMARY,
    use_data_reuploading=USE_REUPLOADING
)

print('Shot ablation results:')
display(df_shots)

---
## Cell 13 — Real-hardware validation (consumes qBraid QPU credits)
**Read this before running.** Cells 1–12 above are 100% simulator, by
design — the sweeps alone (Hamiltonian grid, qubit scaling, noise, shots)
are dozens of configs × hundreds of timesteps each, which would be
thousands of individually-queued hardware jobs if pointed at real hardware.
This cell instead validates ONLY the already-selected config from Cell 8
(J*, h*, p*, topology*, encoding, and the fitted warm-start readout) over a
SMALL subsampled window (`HW_N_STEPS` below) — that's the only part of this
notebook that should touch real QPU credits.

**Backend status:**
- `simulation` (default) — free, no credits used, safe to run any time.
- `aquila` — **PRIMARY.** QuEra Aquila via `bloqade-analog`. Aquila is an
  *analog* Rydberg device with no gate set, so this is NOT a device swap —
  `J`/`h` are re-expressed as a real physical program (atom spacing sets the
  interaction strength, global Rabi drive sets the transverse field,
  per-atom local detuning encodes the classical input). Full reasoning in
  `reservoir/aquila_backend.py`'s module docstring — **read it before
  trusting numbers from this backend.** Validated end-to-end against two
  FREE local emulators (Bloqade's own, and AWS Braket's stricter one) — both
  confirm the generated programs are physically well-formed and produce
  bounded, encoding-sensitive output. **Not yet run against live Aquila
  hardware.** `config.AQUILA_SUBMIT_TARGET` (or the `$AQUILA_SUBMIT_TARGET`
  env var) controls whether this cell's `aquila` selection actually reaches
  real hardware (`"aquila"`) or one of the free emulators
  (`"local_emulator"` default, or `"braket_local_emulator"` for the
  stricter check) — confirm it says `"aquila"` before running this if you
  intend to spend real credits, and start with a tiny `HW_N_STEPS` (5–10).
- `ibm` — **FALLBACK.** Real IBM Eagle/Heron via Qiskit Runtime. A genuine
  device swap (this project's gate-based circuit runs unchanged) but **not
  yet verified against live IBM hardware** either. Requires
  `qiskit-ibm-runtime` + `pennylane-qiskit` and a saved IBM Quantum account
  token.

In [ ]:
# ── HARDWARE BACKEND SELECTION ─────────────────────────────────────────────────
HARDWARE_BACKEND = 'simulation'   # 'simulation' (free) | 'aquila' (PRIMARY, real QPU credits) | 'ibm' (FALLBACK, real QPU credits)
HW_N_STEPS = 10                   # keep SMALL — each step submits real circuit(s) on real hardware

import os
print(f'AQUILA_SUBMIT_TARGET = {os.environ.get("AQUILA_SUBMIT_TARGET", "(from config.py, default local_emulator)")}')

if HARDWARE_BACKEND == 'aquila':
    print('Aquila backend selected. See the markdown above — this consumes real')
    print('QPU credits ONLY if AQUILA_SUBMIT_TARGET="aquila" (config.py or env var).')
    print('Otherwise this runs against a free local emulator as a dry run.')
elif HARDWARE_BACKEND == 'ibm':
    print('IBM backend selected — ensure qiskit-ibm-runtime + pennylane-qiskit are')
    print('installed and an IBM Quantum account token is saved. This WILL submit')
    print('real circuits and consume real QPU time/credits if it succeeds.')
else:
    print('Simulation backend — free, no QPU credits used.')

from scripts.hardware_validation import run_hardware_validation

try:
    hw_result = run_hardware_validation(
        horizon=6, n_steps=HW_N_STEPS, backend=HARDWARE_BACKEND, mode='warm_start_qrc'
    )
    print('\n=== Hardware validation result ===')
    print(json.dumps(hw_result, indent=2))
    if HARDWARE_BACKEND != 'simulation' and HARDWARE_BACKEND in hw_result:
        delta = hw_result.get('sim_vs_hw_rmse_delta')
        print(f'\nSimulator vs {HARDWARE_BACKEND} RMSE delta: {delta:+.4f} '
              f'(positive = hardware worse than simulator)')

except FileNotFoundError as e:
    print(f'Run Cell 8 first — need a fitted warm_start_qrc config + readout: {e}')
except ValueError as e:
    # AquilaBackend geometry errors land here (e.g. n_qubits doesn't fit the
    # lattice at this J) — see reservoir/aquila_backend.py::_compute_geometry
    print(f'Configuration error: {e}')
except ImportError as e:
    print(f'Missing dependency for backend={HARDWARE_BACKEND}: {e}')
except Exception as e:
    import traceback
    print(f'Hardware validation failed: {e}')
    traceback.print_exc()

---
## Cell 14 — Summary and output file list
All output files needed for the write-up are listed here.

In [ ]:
from config import RESULTS
import json

output_files = list(RESULTS.glob('*'))
print(f'Output files in {RESULTS}:')
for f in sorted(output_files):
    size_kb = f.stat().st_size // 1024 if f.exists() else 0
    print(f'  {f.name:<45} {size_kb:>6} KB')

# Print key numbers for write-up
print('\n=== KEY NUMBERS FOR WRITE-UP ===')
try:
    cfg = json.load(open(RESULTS / 'h6' / 'warm_start_qrc_config.json'))
    print(f'QRC config: n={cfg["n_qubits"]} J={cfg["J"]} h={cfg["h"]} '
          f'encoding={"data_reuploading" if cfg["use_data_reuploading"] else "standard"}')
    print(f'Hardware backend used for training: {cfg.get("hardware_backend", "simulation")}')
    print(f'Circuit depth: {cfg["trotter_steps"]} Trotter steps '
          f'(effective depth ~{cfg["trotter_steps"] * cfg["n_qubits"]})')
    print(f'Wall-clock training time: {cfg.get("wall_clock_s", "N/A")}s')
    print(f'Readout strategy: {cfg["readout_strategy"]}')
    print(f'Val RMSE (mean, normalised): {cfg["val_rmse_mean"]:.4f}')
except FileNotFoundError:
    print('(Run Cell 8 first to generate QRC config)')

if (RESULTS / 'hardware_validation.json').exists():
    hw = json.load(open(RESULTS / 'hardware_validation.json'))
    print(f'\nHardware validation (Cell 13): requested_backend={hw.get("requested_backend")}')
    for backend_key in ('simulation', hw.get('requested_backend')):
        if backend_key and backend_key in hw:
            print(f'  {backend_key}: rmse_mean={hw[backend_key]["rmse_mean"]:.4f} '
                  f'n_steps={hw[backend_key]["n_steps"]} wall_clock={hw[backend_key]["wall_clock_s"]}s')
else:
    print('\n(Run Cell 13 to generate a real/simulated hardware validation data point)')

print('\nPhase 3 checklist:')
checks = [
    ('results_h6.csv exists',    (RESULTS / 'results_h6.csv').exists()),
    ('results_h24.csv exists',   (RESULTS / 'results_h24.csv').exists()),
    ('qubit_scaling.csv exists', (RESULTS / 'qubit_scaling.csv').exists()),
    ('noise_sweep.csv exists',   (RESULTS / 'noise_sweep.csv').exists()),
    ('shot_ablation.csv exists', (RESULTS / 'shot_ablation.csv').exists()),
    ('encoding_ablation.json',   (RESULTS / 'encoding_ablation.json').exists()),
    ('warm_start_qrc_config (h6)', (RESULTS / 'h6' / 'warm_start_qrc_config.json').exists()),
    ('warm_start_qrc_readout (h6)', (RESULTS / 'h6' / 'warm_start_qrc_readout.pkl').exists()),
    ('hardware_validation.json (Cell 13)', (RESULTS / 'hardware_validation.json').exists()),
]
for name, ok in checks:
    print(f'  {"✓" if ok else "✗"} {name}')